# 07 — Document Labels: Industry, Direction, Mechanism, Adoption

`pipeline/s50_doc_labels.py` draws a quarter-stratified random sample of about 15,000 analysis articles (15,001 after per-quarter rounding) and labels each with one structured LLM prompt (`pipeline/taxonomy.py` holds the label sets):

- **industries**: up to two of 15 sectors, or none when the article is about AI in general;
- **direction**: net effect of AI on the first industry (positive / negative / mixed / neutral);
- **mechanisms**: automation, augmentation, cost & efficiency, new offerings, workflow redesign — only when stated;
- **outcome**: deployed, pilot or plan, setback, vendor product launch, or none;
- **factors**: enablers and barriers named in the article (data, cost/ROI, regulation, talent, integration, trust & safety, compute, partnerships, leadership).

The prompt contains only the article title and text (no model scores), so these labels are independent of the 06C sentiment estimates.

Industry labels are extended to all analysis articles with one-vs-rest logistic regressions on mean block embeddings from 04; thresholds are tuned on a validation split and scored on a held-out test split.

In [1]:
import os, sys

PROJECT_DIR = "/content/drive/MyDrive/NLP/NLP_FINAL_PROJECT_Tom_Chen"
IN_COLAB = os.path.isdir("/content")
if IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.environ.setdefault("PROJECT_DIR", PROJECT_DIR)
if IN_COLAB and "DEEPSEEK_API_KEY" not in os.environ:
    from google.colab import userdata
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY").strip()
# Path to a checkout of this repository (on Drive in Colab).
sys.path.insert(0, os.environ.get("REPO_DIR", f"{os.environ['PROJECT_DIR']}/code"))

import json
import pandas as pd
from pipeline import config

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [2]:
RUN = False  # set True to relabel (~15K LLM calls, about $1.7 off-peak) and retrain the classifier
if RUN:
    from pipeline import s50_doc_labels
    s50_doc_labels.main()

In [3]:
lab = pd.read_parquet(config.out("industry", "doc_labels_sample.parquet"))
print(f"labeled articles: {len(lab):,}")
for col in ["industry_primary", "direction", "outcome"]:
    print(lab[col].value_counts(normalize=True).round(3).to_string(), "\n")

labeled articles: 15,001
industry_primary
software_it                  0.352
none                         0.137
media_entertainment          0.112
healthcare_life_sciences     0.066
semiconductors_hardware      0.060
government_defense           0.057
education                    0.046
finance_insurance            0.045
retail_consumer              0.035
transportation_automotive    0.022
legal_professional           0.020
energy_utilities             0.013
manufacturing_industrial     0.012
telecom                      0.011
agriculture_food             0.007
real_estate_construction     0.006 

direction
positive    0.523
neutral     0.235
mixed       0.164
negative    0.079 

outcome
none              0.343
product_launch    0.339
deployed          0.145
pilot_or_plan     0.123
setback           0.050 



## Examples

In [4]:
docs = pd.read_parquet(config.out("corpus", "docs.parquet"), columns=["doc_id", "title"])
lab.merge(docs, on="doc_id").sample(12, random_state=1)[["title", "industries", "direction", "mechanisms", "outcome"]]

,title,industries,direction,mechanisms,outcome
11779,The Impending AI Bubble Burst: Steve Keen’s Warnings and Concurring Voices: part 2,software_it|finance_insurance,negative,,none
10510,Microsoft Sets 16GB RAM & 40 TOPS of AI Computing Speed as Standard for New 'AI PCs' | Beebom,semiconductors_hardware|software_it,neutral,,pilot_or_plan
10310,"Alan AI Assessed ""Awardable"" for Department of Defense Work in the CDAO's Tradewinds Solutions Marketplace",government_defense|software_it,positive,automation|workflow_redesign,product_launch
5151,NoBroker unveils ConvoZen.AI: A comprehensive conversational AI cloud for the agentic era - CRN - India,software_it|real_estate_construction,positive,augmentation|automation|cost_efficiency,product_launch
506,New Study Reveals the “New Normal” in Finance: Economic Pressures Push Middle Market Teams Toward Faster Digital and...,finance_insurance,positive,automation|cost_efficiency|workflow_redesign,pilot_or_plan
1622,‘Trust your abilities’: How this linguist found a path to data science,,neutral,,none
12918,Chatting with Meta's dystopian AI personas leaves more questions than answers | Mashable,software_it|media_entertainment,negative,,product_launch
2145,OpenAI Establishes Its First Asian Hub in Tokyo,software_it,positive,,product_launch
10963,Proggio Unveils Contextual Agentic AI Enabling Project Teams to Move Beyond Planning and Into Execution,software_it,positive,augmentation|automation|workflow_redesign,product_launch
4528,The silent AI risk inside your business: Why AI governance must come first | ITWeb,software_it,mixed,augmentation|workflow_redesign,pilot_or_plan


## Industry classifier on held-out LLM labels

In [5]:
metrics = json.load(open(config.out("industry", "industry_classifier_metrics.json")))
pd.DataFrame(metrics).T[["n_pos", "threshold", "test_precision", "test_recall", "test_f1", "test_ap",
                         "test_prevalence_llm", "test_prevalence_pred"]].astype(float).round(3).sort_values("test_f1")

,n_pos,threshold,test_precision,test_recall,test_f1,test_ap,test_prevalence_llm,test_prevalence_pred
telecom,307.0,0.825,0.425,0.574,0.488,0.574,0.024,0.032
manufacturing_industrial,342.0,0.925,0.548,0.469,0.505,0.525,0.022,0.019
legal_professional,514.0,0.925,0.677,0.489,0.568,0.664,0.040,0.029
energy_utilities,311.0,0.800,0.484,0.769,0.594,0.721,0.017,0.028
finance_insurance,934.0,0.925,0.817,0.508,0.626,0.694,0.059,0.036
real_estate_construction,128.0,0.875,0.591,0.684,0.634,0.724,0.008,0.010
government_defense,1330.0,0.900,0.747,0.565,0.644,0.734,0.095,0.072
retail_consumer,703.0,0.825,0.618,0.680,0.648,0.719,0.044,0.049
agriculture_food,135.0,0.950,0.789,0.556,0.652,0.755,0.012,0.008
transportation_automotive,459.0,0.850,0.703,0.692,0.698,0.787,0.029,0.028
